In [ ]:
import pandas as pd

# Determines the top 4/5 songs per decade based on charts.csv 
# using how many times a song shows up and its rank

## Load the data into df
df = pd.read_csv("charts.csv")

# Clean data
df['date'] = pd.to_datetime(df['date'])
df['decade'] = (df['date'].dt.year // 10 * 10).astype(str) + 's'

# Aggregate per song+artist per decade
song_decade = df.groupby(['decade', 'song', 'artist']).agg(
    appearances=('rank', 'count'),
    avg_rank=('rank', 'mean'),
    peak=('rank', 'min') 
).reset_index()

# Score = appearances × (101 - avg_rank)
# More weeks + better average rank = higher score
song_decade['score'] = song_decade['appearances'] * (101 - song_decade['avg_rank'])

# Get top 5 from each decade based on highest score
top5 = (
    song_decade
    .sort_values('score', ascending=False)
    .groupby('decade')
    .head(5)
    .sort_values(['decade', 'score'], ascending=[True, False])
    .reset_index(drop=True)
)

# Write to csv
top5[['decade','song','artist','appearances','avg_rank','peak','score']].to_csv('top5_by_decade.csv', index=False)

